# OpenAI scratch notebook

Paste your API key and a prompt string, then run the cells.

Defaults match the project's `llm.py` (LiteLLM proxy, `gpt-5.4`, temperature 0). No extra dependencies — uses `urllib` from the stdlib.

In [13]:
import os
API_KEY  = os.getenv('OPENAI_API_KEY')
ENDPOINT = "http://dep-eng-data-s-heimgarten.hosts.utn.de:4000/v1/chat/completions"
MODEL    = "gpt-5.4"  # e.g. "gpt-5.4-mini"
TEMPERATURE = 0

In [14]:
import json
import urllib.request


def ask(prompt, system=None, model=MODEL, temperature=TEMPERATURE, timeout=90):
    """Send a single chat completion request and return the assistant text."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    payload = json.dumps({
        "model": model,
        "messages": messages,
        "temperature": temperature,
    }).encode("utf-8")

    request = urllib.request.Request(
        ENDPOINT,
        data=payload,
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {API_KEY}",
        },
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=timeout) as resp:
        data = json.loads(resp.read().decode("utf-8"))
    return data["choices"][0]["message"]["content"]

In [ ]:
# --- Paste the string you want to send here ---
PROMPT = """
Repair faulty CSV records. Return JSON only, no Markdown, no commentary.
For each input line, return the respective field values, not CSV text.
Do not CSV-escape inside field values; JSON escaping is allowed only because the response is JSON.

Use the good examples as a guide for alignment, order of the data cells by looking at patterns and semantics.
Don't do cleaning of cell data, so keep accidental double whitespaces, typos etc.
Dialect delimiters at the start or end of a row as well as double delimiters in the middle of a row denote empty fields. These should be preserved and included as an empty string in your output list. Only drop them if they shift the columns in a way that would not be consistent with the example rows or that would break the expected number of columns.
Keep in mind that literal quote characters in field text are escaped with the escape character that can also be a quote. In this case, keep one as literal field text. 

Priority order: exactly hit the expected column count; each value fits its column semantically (e.g. from header or example rows); preserve original cell text.
Priority order: exactly hit the expected column count; each value fits its column semantically (e.g. from header or example rows); preserve original cell text.

Response shape:
{"repairs": [{"line": <line number>, "fields": [<string>, ...]}]}

Input:
{"expected_column_count": 9, "columns": ["DATE", "TIME", "Qty", "PRODUCTID", "Price", "ProductType", "ProductDescription", "URL", "Comments"], "dialect": {"delimiter": ",", "quotechar": "\"", "escapechar": "\"", "row_delimiter": null, "header_lines": 1, "preamble_lines": 0}, "good_examples": [{"fields": ["28/01/2018", "00:00", "2", "MG-8769", "$74.69", "Men's Waterproof Hiking Boots", "These waterproof hiking boots for men are rugged enough for peak performance yet light and quick enough to keep feet from feeling weighed down.", "https://www.example.com/product/MG_8769.html", ""]}, {"fields": ["29/01/2018", "00:15", "0", "RI-3895", "$29.81", "Light-Up Running Jacket", "The next level of weather protection. This light-up jacket resists the elements and keeps you visible in low-light conditions. From running, biking or walking the dog, the durable construction and innovative safety features won't let you down.", "https://www.example.com/product/RI_3895.html", ""]}, {"fields": ["31/01/2018", "00:45", "1", "RI-9546", "$25.55", "Switch Fly Rods", "This lightweight fly rod delivers outstanding performance and can be used as either a traditional one-handed rod or as a two-handed spey rod. Two-handed technique is ideal for larger rivers and situations where there isn't space for a backcast.", "https://www.example.com/product/RI_9546.html", ""]}, {"fields": ["13/02/2018", "01:00", "9", "CC-9259", "$48.00", "ThrowPillow, Wooden Paddles", "Add a pop of paddling fun to your bed, chair or sofa with this whimsical throw pillow, handhooked on front for a timeless style.", "https://www.example.com/product/CC_9259.html", ""]}, {"fields": ["14/02/2018", "01:15", "1", "CC-1697", "$34.22", "Men's Heavy-Duty Suspenders", "These tough Men's Heavy-Duty Suspenders are made to hold up heavy wool pants without stretching in any way, shape or form.", "https://www.example.com/product/CC_1697.html", ""]}, {"fields": ["15/02/2018", "01:30", "2", "RI-6052", "$89.34", "Organic Textured Cotton Towel", "All the softness and absorbency you've come to expect from our towels, in certified organic cotton for natural, ecofriendly comfort.", "https://www.example.com/product/RI_6052.html", ""]}, {"fields": ["16/02/2018", "01:45", "2", "YY-3522", "$19.34", "Cycling Jersey, Short-Sleeve", "Designed with lots of performance features, plus a semi-form-fitting profile, this cycling jersey delivers all-day comfort and serious style.", "https://www.example.com/product/YY_3522.html", ""]}, {"fields": ["17/02/2018", "02:00", "0", "YY-5315", "$45.39", "Men's Silk Underwear, Crewneck", "For strong, lightweight comfort without bulk, our men's silk crewneck makes for an ideal first layer against the cold from slope to lodge to shoveling snow.", "https://www.example.com/product/YY_5315.html", ""]}, {"fields": ["18/02/2018", "02:15", "2", "BH-9827", "$78.07", "All-Weather Dining Table, Round 48\"", "Made in the USA to our exacting standards, this round patio table is durable enough to weather the elements year-round.", "https://www.example.com/product/BH_9827.html", ""]}, {"fields": ["19/02/2018", "02:30", "1", "BH-7885", "$52.45", "Women's No-Show Socks", "The warmth and comfort of wool, these socks are designed in a minimal, no-show style that is of pure elegance and design.", "https://www.example.com/product/BH_7885.html", ""]}], "faulty_lines": [{"line": 4, "raw": "30/01/2018,00:30,1,RI-8070,$80.08,\"Men's Ventilated Trail Shoes,\"Great grip and super extra breathability make these amazing ventilated hikers ideal for warm, dry conditions.\",\"https://www.example.com/product/RI_8070.html\",", "errors": [{"type": "UNQUOTED VALUE", "message": "Value with unterminated quote found.", "column_idx": 6, "column_name": "_c5"}]}]}
"""

print(ask(PROMPT))

{"repairs":[{"line":4,"fields":["30/01/2018","00:30","1","RI-8070","$80.08","Men's Ventilated Trail Shoes","Great grip and super extra breathability make these amazing ventilated hikers ideal for warm, dry conditions.","https://www.example.com/product/RI_8070.html",""]}]}


In [ ]:
import dialect

